In [0]:
# Section 1 — Read Bronze Sales Details

df = spark.table("bike_lakehouse.bronze.crm_sales_details")

display(df)

In [0]:
# Section 2 — Inspect Schema

df.printSchema()

In [0]:
# Section 3 — Basic Data Quality Profile

from pyspark.sql.functions import col

print("Total rows:", df.count())

for column_name in df.columns:
    print(
        column_name,
        "NULLs:",
        df.filter(col(column_name).isNull()).count()
    )

In [0]:
# Section 4 — Inspect Missing Sales and Price

display(
    df.filter(
        col("sls_sales").isNull() |
        col("sls_price").isNull()
    )
)

In [0]:
# Section 5 — Validate Sales / Quantity / Price Relationship

display(
    df.filter(
        col("sls_sales").isNotNull() &
        col("sls_quantity").isNotNull() &
        col("sls_price").isNotNull()
    )
    .withColumn(
        "expected_sales",
        col("sls_quantity") * col("sls_price")
    )
    .filter(
        col("sls_sales") != col("expected_sales")
    )
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_quantity",
        "sls_price",
        "sls_sales",
        "expected_sales"
    )
)

In [0]:
# Section 6 — Inspect Invalid Numeric Values

display(
    df.filter(
        (col("sls_price") <= 0) |
        (col("sls_sales") <= 0) |
        (col("sls_quantity") <= 0)
    )
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_cust_id",
        "sls_quantity",
        "sls_price",
        "sls_sales"
    )
)

In [0]:
# Section 7 — Convert Integer Dates to DATE

from pyspark.sql.functions import to_date

df_clean = (
    df
    .withColumn(
        "sls_order_dt",
        to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")
    )
    .withColumn(
        "sls_ship_dt",
        to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")
    )
    .withColumn(
        "sls_due_dt",
        to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")
    )
)

df_clean.printSchema()

In [0]:
# Section 8 — Validate Date Relationships

display(
    df_clean.filter(
        (col("sls_ship_dt") < col("sls_order_dt")) |
        (col("sls_due_dt") < col("sls_ship_dt"))
    )
    .select(
        "sls_ord_num",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt"
    )
)

In [0]:
# Section 8 — Find Invalid Date Values

display(
    df.filter(
        (col("sls_order_dt") == 0) |
        (col("sls_ship_dt") == 0) |
        (col("sls_due_dt") == 0)
    )
)

In [0]:
# Section 9 — Convert Dates Safely

from pyspark.sql.functions import to_date, when

df_clean = (
    df
    .withColumn(
        "sls_order_dt",
        when(
            col("sls_order_dt") == 0,
            None
        ).otherwise(
            to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")
        )
    )
    .withColumn(
        "sls_ship_dt",
        to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd")
    )
    .withColumn(
        "sls_due_dt",
        to_date(col("sls_due_dt").cast("string"), "yyyyMMdd")
    )
)

df_clean.printSchema()

In [0]:
# Section 10 — Verify Invalid Order Dates

print(
    "NULL order dates:",
    df_clean.filter(col("sls_order_dt").isNull()).count()
)

display(
    df_clean
    .filter(col("sls_order_dt").isNull())
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt"
    )
)

In [0]:
# Section 9 — Convert Dates Safely

from pyspark.sql.functions import try_to_date

df_clean = (
    df
    .withColumn(
        "sls_order_dt",
        try_to_date(
            col("sls_order_dt").cast("string"),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "sls_ship_dt",
        try_to_date(
            col("sls_ship_dt").cast("string"),
            "yyyyMMdd"
        )
    )
    .withColumn(
        "sls_due_dt",
        try_to_date(
            col("sls_due_dt").cast("string"),
            "yyyyMMdd"
        )
    )
)

df_clean.printSchema()

In [0]:
# Section 10 — Date Quality Check

for column_name in ["sls_order_dt", "sls_ship_dt", "sls_due_dt"]:
    print(
        column_name,
        "NULL/invalid dates:",
        df_clean.filter(col(column_name).isNull()).count()
    )

In [0]:
# Section 11 — Inspect Invalid Order Dates

display(
    df.filter(
        (col("sls_order_dt") == 0) |
        (col("sls_order_dt") == 32154) |
        col("sls_order_dt").isNull()
    )
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_cust_id",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt",
        "sls_sales",
        "sls_quantity",
        "sls_price"
    )
)

In [0]:
# Section 12 — Validate Date Relationships

display(
    df_clean
    .filter(
        col("sls_order_dt").isNotNull() &
        (
            (col("sls_ship_dt") < col("sls_order_dt")) |
            (col("sls_due_dt") < col("sls_ship_dt"))
        )
    )
    .select(
        "sls_ord_num",
        "sls_prd_key",
        "sls_order_dt",
        "sls_ship_dt",
        "sls_due_dt"
    )
)

In [0]:
# Section 13 — Check Exact Duplicate Rows

print(
    "Total rows:",
    df_clean.count()
)

print(
    "Exact duplicate rows:",
    df_clean.groupBy(df_clean.columns)
            .count()
            .filter(col("count") > 1)
            .count()
)

In [0]:
# Section 14 — Check Order + Product Business Key

display(
    df_clean
    .groupBy("sls_ord_num", "sls_prd_key")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

In [0]:
# Section 15 — Business Key NULL Check

for column_name in [
    "sls_ord_num",
    "sls_prd_key",
    "sls_cust_id",
    "sls_quantity"
]:
    print(
        column_name,
        "NULLs:",
        df_clean.filter(col(column_name).isNull()).count()
    )

In [0]:
# Section 16 — Product Key Integrity

sales_products = (
    df_clean
    .select("sls_prd_key")
    .distinct()
)

silver_products = (
    spark.table("bike_lakehouse.silver.crm_products")
    .select("product_key")
    .distinct()
)

unmatched_products = (
    sales_products
    .join(
        silver_products,
        sales_products.sls_prd_key == silver_products.product_key,
        "left_anti"
    )
)

print("Distinct sales product keys:", sales_products.count())
print("Unmatched product keys:", unmatched_products.count())

display(unmatched_products)

In [0]:
# Section 17 — Inspect Unmatched Product Keys

display(
    unmatched_products
    .orderBy("sls_prd_key")
)

In [0]:
# Section 17 — Compare Product Key Counts

print(
    "Sales distinct product keys:",
    sales_products.count()
)

print(
    "Silver distinct product keys:",
    silver_products.count()
)

In [0]:
# Section 18 — Count Matching Product Keys

matched_products = (
    sales_products
    .join(
        silver_products,
        sales_products.sls_prd_key == silver_products.product_key,
        "inner"
    )
)

print(
    "Matching product keys:",
    matched_products.count()
)

display(
    matched_products.orderBy("sls_prd_key")
)

In [0]:
# Section 19 — Inspect Silver Product Keys

display(
    silver_products
    .orderBy("product_key")
    .limit(50)
)

In [0]:
# Section 20 — Test Product Key Relationship

matched_products = (
    sales_products.alias("s")
    .join(
        silver_products.alias("p"),
        col("p.product_key").endswith(col("s.sls_prd_key")),
        "inner"
    )
)

print(
    "Sales keys matched through suffix:",
    matched_products.select("s.sls_prd_key").distinct().count()
)

display(
    matched_products
    .select(
        col("s.sls_prd_key").alias("sales_product_key"),
        col("p.product_key").alias("crm_product_key")
    )
    .orderBy("sales_product_key")
)

In [0]:
# Section 21 — Final Product Referential Integrity Check

print(
    "Sales distinct product keys:",
    sales_products.count()
)

print(
    "Matched product keys:",
    matched_products
        .select("sls_prd_key")
        .distinct()
        .count()
)

print(
    "Unmatched product keys:",
    sales_products
        .join(
            matched_products
                .select(col("sls_prd_key").alias("matched_key"))
                .distinct(),
            sales_products.sls_prd_key == col("matched_key"),
            "left_anti"
        )
        .count()
)

In [0]:
# Section 22 — Customer Key Integrity

sales_customers = (
    df_clean
    .select("sls_cust_id")
    .distinct()
)

silver_customers = (
    spark.table("bike_lakehouse.silver.crm_customers")
    .select("customer_id")
    .distinct()
)

unmatched_customers = (
    sales_customers
    .join(
        silver_customers,
        sales_customers.sls_cust_id == silver_customers.customer_id,
        "left_anti"
    )
)

print("Distinct sales customer IDs:", sales_customers.count())
print("Unmatched customer IDs:", unmatched_customers.count())

display(unmatched_customers)

In [0]:
# Section 23 — Final Customer Referential Integrity Check

print(
    "Sales distinct customer IDs:",
    sales_customers.count()
)

print(
    "Silver distinct customer IDs:",
    silver_customers.count()
)

print(
    "Unmatched customer IDs:",
    unmatched_customers.count()
)

In [0]:
# Section 24 — Numeric Quality Summary

print(
    "NULL sales:",
    df_clean.filter(col("sls_sales").isNull()).count()
)

print(
    "NULL price:",
    df_clean.filter(col("sls_price").isNull()).count()
)

print(
    "Zero sales:",
    df_clean.filter(col("sls_sales") == 0).count()
)

print(
    "Negative sales:",
    df_clean.filter(col("sls_sales") < 0).count()
)

print(
    "Zero price:",
    df_clean.filter(col("sls_price") == 0).count()
)

print(
    "Negative price:",
    df_clean.filter(col("sls_price") < 0).count()
)

print(
    "Zero quantity:",
    df_clean.filter(col("sls_quantity") == 0).count()
)

print(
    "Negative quantity:",
    df_clean.filter(col("sls_quantity") < 0).count()
)

In [0]:
# Section 25 — Rename Columns

df_silver = (
    df_clean
    .withColumnRenamed("sls_ord_num", "order_number")
    .withColumnRenamed("sls_prd_key", "product_key")
    .withColumnRenamed("sls_cust_id", "customer_id")
    .withColumnRenamed("sls_order_dt", "order_date")
    .withColumnRenamed("sls_ship_dt", "ship_date")
    .withColumnRenamed("sls_due_dt", "due_date")
    .withColumnRenamed("sls_sales", "sales_amount")
    .withColumnRenamed("sls_quantity", "quantity")
    .withColumnRenamed("sls_price", "unit_price")
)

df_silver.printSchema()

In [0]:
# Section 26 — Final Silver Sanity Check

print("Final row count:", df_silver.count())

print(
    "Duplicate business keys:",
    df_silver
    .groupBy("order_number", "product_key")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("NULL order numbers:",
      df_silver.filter(col("order_number").isNull()).count())

print("NULL product keys:",
      df_silver.filter(col("product_key").isNull()).count())

print("NULL customer IDs:",
      df_silver.filter(col("customer_id").isNull()).count())

print("NULL order dates:",
      df_silver.filter(col("order_date").isNull()).count())

print("NULL ship dates:",
      df_silver.filter(col("ship_date").isNull()).count())

print("NULL due dates:",
      df_silver.filter(col("due_date").isNull()).count())

In [0]:
# Section 27 — Write Sales Silver Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.crm_sales_details")

print("Sales Silver table created successfully.")